# 01 - Data Acquisition

**Purpose:** Download all 5 configured datasets, verify them, and record provenance in MANIFEST.json.

**Inputs:** configs/datasets.yaml

**Outputs:** Raw files under data/raw/, updated data/MANIFEST.json, a provenance table.

**Runtime estimate:** 5-30 minutes depending on network speed (several hundred MB total).

## Load dataset configuration

In [1]:
from pathlib import Path

from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_raw_dir

datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]
raw_dir = data_raw_dir()
raw_dir.mkdir(parents=True, exist_ok=True)
print(f"{len(datasets_cfg)} datasets configured: {list(datasets_cfg.keys())}")


5 datasets configured: ['household_power_consumption', 'appliances_energy_prediction', 'air_quality_uci', 'beijing_multisite_air_quality', 'jena_climate']


## Download each dataset (ucimlrepo then direct-URL fallback)

In [2]:
import warnings

import pandas as pd

from sdg_leakage_audit.data.download import download_file, try_ucimlrepo_then_direct
from sdg_leakage_audit.utils.logging import get_logger
from sdg_leakage_audit.utils.paths import to_project_relative

logger = get_logger("notebook01")

# household_power_consumption has mixed-type columns read by ucimlrepo's internal
# pd.read_csv; this is informational only (we re-type columns in notebook 02) and the
# warning text embeds an absolute path to the installed package file, so suppress it.
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

download_results = {}  # dataset_id -> {"local_path": Path, "resolved_route": str, "url": str, "error": str | None}

for dataset_id, cfg in datasets_cfg.items():
    source = cfg["source"]
    uci_id = cfg.get("uci_id")
    try:
        if uci_id is not None:
            local_path, resolved_route = try_ucimlrepo_then_direct(
                uci_id=uci_id,
                direct_url=source["direct_zip_url"],
                dest_dir=raw_dir,
            )
            configured_url = source.get("ucimlrepo", source["direct_zip_url"])
        else:
            # Non-UCI source (Jena Climate): direct download only.
            dest_path = raw_dir / f"{dataset_id}.csv.zip"
            local_path = download_file(source["direct_zip_url"], dest_path)
            resolved_route = "direct_zip_url"
            configured_url = source["direct_zip_url"]

        download_results[dataset_id] = {
            "local_path": local_path,
            "resolved_route": resolved_route,
            "url": configured_url,
            "error": None,
        }
        logger.info(
            "OK  %-32s via %-14s -> %s",
            dataset_id,
            resolved_route,
            to_project_relative(local_path),
        )
    except Exception as exc:  # noqa: BLE001 - record and continue with remaining datasets
        download_results[dataset_id] = {
            "local_path": None,
            "resolved_route": None,
            "url": source.get("direct_zip_url"),
            "error": str(exc),
        }
        logger.error("FAIL %-32s : %s", dataset_id, exc)

failed = {k: v for k, v in download_results.items() if v["error"] is not None}
if failed:
    print(f"WARNING: {len(failed)} dataset(s) failed to download: {list(failed.keys())}")
else:
    print("All datasets downloaded successfully.")


2026-10-21 20:28:08 | INFO     | sdg_leakage_audit.data.download | Attempting ucimlrepo route for UCI id=235


2026-10-21 20:33:51 | INFO     | sdg_leakage_audit.data.download | ucimlrepo route succeeded for UCI id=235 -> data\raw\uci_235_ucimlrepo.csv (2075259 rows)


2026-10-21 20:33:51 | INFO     | notebook01 | OK  household_power_consumption      via ucimlrepo      -> data\raw\uci_235_ucimlrepo.csv


2026-10-21 20:33:51 | INFO     | sdg_leakage_audit.data.download | Attempting ucimlrepo route for UCI id=374


2026-10-21 20:33:56 | INFO     | sdg_leakage_audit.data.download | ucimlrepo route succeeded for UCI id=374 -> data\raw\uci_374_ucimlrepo.csv (19735 rows)


...

2026-10-21 20:34:05 | INFO     | sdg_leakage_audit.data.download | Downloaded https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip -> data\raw\uci_501.zip (8192212 bytes)


2026-10-21 20:34:05 | INFO     | notebook01 | OK  beijing_multisite_air_quality    via direct_zip_url -> data\raw\uci_501.zip


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.download | Downloaded https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip -> data\raw\jena_climate.csv.zip (13568290 bytes)


2026-10-21 20:34:09 | INFO     | notebook01 | OK  jena_climate                     via direct_zip_url -> data\raw\jena_climate.csv.zip


All datasets downloaded successfully.


## Verify downloads and update MANIFEST.json

In [3]:
from sdg_leakage_audit.data.manifest import load_manifest, save_manifest, update_manifest
from sdg_leakage_audit.utils.paths import to_project_relative

manifest = load_manifest()

for dataset_id, result in download_results.items():
    if result["error"] is not None:
        continue
    local_path = result["local_path"]
    if not local_path.is_file() or local_path.stat().st_size == 0:
        print(f"VERIFY FAILED: {dataset_id} -> {to_project_relative(local_path)} missing or empty")
        continue
    manifest = update_manifest(
        dataset_id=dataset_id,
        url=result["url"],
        resolved_url=result["resolved_route"],
        local_path=local_path,
    )

print(f"Manifest now has {len(manifest['datasets'])} dataset entries.")


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.manifest | Manifest written to data\MANIFEST.json (5 dataset entries)


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.manifest | Manifest written to data\MANIFEST.json (5 dataset entries)


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.manifest | Manifest written to data\MANIFEST.json (5 dataset entries)


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.manifest | Manifest written to data\MANIFEST.json (5 dataset entries)


2026-10-21 20:34:09 | INFO     | sdg_leakage_audit.data.manifest | Manifest written to data\MANIFEST.json (5 dataset entries)


Manifest now has 5 dataset entries.


## Provenance summary table

In [4]:
import pandas as pd

manifest = load_manifest()
rows = []
for dataset_id, entry in manifest["datasets"].items():
    cfg = datasets_cfg.get(dataset_id, {})
    rows.append(
        {
            "dataset_id": dataset_id,
            "name": cfg.get("name"),
            "sdg_tag": cfg.get("sdg_tag"),
            "resolved_route": entry["resolved_url"],
            "size_bytes": entry["size_bytes"],
            "sha256": entry["sha256"][:12] + "...",
            "date_downloaded": entry["date_downloaded"],
        }
    )

provenance_df = pd.DataFrame(rows)
provenance_df


,dataset_id,name,sdg_tag,resolved_route,size_bytes,sha256,date_downloaded
0,household_power_consumption,Individual household electric power consumption,SDG7,ucimlrepo,128517744,d3a1571b7300...,2026-10-21T15:04:09.613308+00:00
1,appliances_energy_prediction,Appliances energy prediction,SDG7,ucimlrepo,4427476,fdf5cd6550d1...,2026-10-21T15:04:09.804841+00:00
2,air_quality_uci,Air Quality,SDG3_SDG11,ucimlrepo,756065,d8196847f231...,2026-10-21T15:04:09.846351+00:00
3,beijing_multisite_air_quality,Beijing Multi-Site Air-Quality Data,SDG3_SDG11,direct_zip_url,8192212,b04da438b2f3...,2026-10-21T15:04:09.866634+00:00
4,jena_climate,Jena Climate 2009-2016,SDG13,direct_zip_url,13568290,63d757501e92...,2026-10-21T15:04:09.919181+00:00


In [5]:
from sdg_leakage_audit.utils.paths import results_tables_dir, to_project_relative

out_path = results_tables_dir() / "data_provenance.csv"
provenance_df.to_csv(out_path, index=False)
print(f"Saved provenance table -> {to_project_relative(out_path)}")


Saved provenance table -> results\tables\data_provenance.csv


## Findings / what to carry forward

- All 5 configured datasets were downloaded and verified; see `results/tables/data_provenance.csv`
  and `data/MANIFEST.json` for exact resolved routes, sizes, and SHA-256 hashes.
- UCI datasets were fetched via the `ucimlrepo` package route (features+targets exported to CSV);
  the direct-URL fallback was not needed but remains available if the package route breaks.
- Jena Climate was fetched via direct download of the Google-mirrored zip.
- Carry forward to notebook 02: raw file formats differ per dataset (ucimlrepo CSV export vs.
  original UCI zip vs. Jena zip) — cleaning/loading code must handle each dataset's raw layout
  (e.g. Date+Time split columns, year/month/day/hour columns for Beijing, -200 sentinel in Air
  Quality) individually in `src/sdg_leakage_audit/data/loaders.py` and `cleaning.py`.
